# Mini Project Part B – M&A Knowledge Assistant using LangChain + RAG (Solution)

In [ ]:
!pip install langchain langchain-google-genai langchain-community langchain-core langchain-text-splitters pypdf faiss-cpu

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
import os
import getpass

In [ ]:
os.environ["GEMINI_API_KEY"]=getpass.getpass("Enter your API Key")

In [ ]:
llm=ChatGoogleGenerativeAI(
    model='gemini-3.6-flash',
    temperature=0 #deterministic answers -> less hallucination
)

## Task 1 — Load and Process the PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("/content/M&A_Due_Diligence_Playbook.pdf")

In [ ]:
documents=loader.load()

report=""
for page in documents:
  report += page.page_content + '\n'

print("Total number of pages",len(documents))
print("Total characters",len(report))

In [ ]:
print(report[:1000]) #first 1000 characters from the document

### Chunking

Experiment with different chunk sizes and overlaps and compare the number of chunks produced.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter #helps in chunking

for size,overlap in [(300,50),(500,100),(1000,200),(2000,400)]:
  splitter=RecursiveCharacterTextSplitter(chunk_size=size,chunk_overlap=overlap)
  c=splitter.split_documents(documents)
  print(f"chunk_size={size:4d} | overlap={overlap:3d} | chunks={len(c):3d} | avg length={sum(len(x.page_content) for x in c)//len(c)}")

The playbook is short and organised in small bullet-style sections (clauses, valuation methods, risks), so we use **chunk_size=500** characters with **chunk_overlap=100**. This keeps one clause or method (heading + description) inside one chunk while the overlap preserves context across chunk boundaries.

In [ ]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500, #max characters in each chunk
    chunk_overlap=100 #characters overlapping between consecutive chunks
)

chunks=text_splitter.split_documents(documents)
print("Number of chunks:",len(chunks))
print("Avg chunk length:",sum(len(c.page_content) for c in chunks)//len(chunks))
print("\nSample chunk (page",chunks[3].metadata['page']+1,"):\n",chunks[3].page_content)

### Questions

1. **Why is document chunking required?** LLMs and embedding models have limited input size, and a whole document embedded as one vector is too vague to match a specific question. Chunking splits the text into focused pieces so that only the relevant parts are retrieved and sent to the LLM (lower cost, better accuracy).
2. **What happens if chunks are too small?** They lose context (a clause may be split from its explanation), embeddings carry little meaning, and the LLM receives fragments, so answers become incomplete.
3. **What happens if chunks are too large?** They mix several topics, so embeddings become diluted and retrieval is less precise; they also use more tokens and add irrelevant text that can distract the LLM.
4. **How did you select your chunk size?** By experimenting with several sizes (above). Since the playbook is short with compact sections, 500 characters with 100 overlap keeps each section intact without mixing topics.

## Task 2 — Generate Embeddings

- **Embedding model**: gemini-embedding-2 (Google Gemini)

1. **What is an embedding?** A dense numerical vector that represents the meaning of a piece of text; texts with similar meaning get vectors that are close together.
2. **Why are embeddings required for RAG?** They let us search by meaning instead of exact keywords, so the question can be matched with the most relevant playbook chunks even when the wording differs.
3. **How does semantic similarity work in retrieval?** The question is converted to a vector using the same embedding model, and its distance/similarity (e.g. cosine similarity or L2 distance) to every chunk vector is computed. The closest chunks are the most semantically similar and are returned as the context.

In [ ]:
embeddings=GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-2'
)

sample_vector=embeddings.embed_query(chunks[0].page_content)
print("Embedding dimension:",len(sample_vector))

## Task 3 — Build a Vector Store

FAISS vector store built only from the chunks of the M&A PDF.

In [ ]:
from langchain_community.vectorstores import FAISS
vector_store=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print("Vectors stored:",vector_store.index.ntotal)

In [ ]:
retriever=vector_store.as_retriever(
    search_type="similarity", #similarity search between query embedding and chunk embeddings
    search_kwargs={"k":3} #top 3 chunks
)

## Task 4 — Build the RAG Pipeline

## Task 5 — RAG Prompt Engineering

The prompt instructs the LLM to:
1. Use **only** the retrieved context.
2. Avoid hallucinating (no outside knowledge).
3. Clearly state when the information is unavailable.
4. Give a concise but complete answer.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
rag_prompt=ChatPromptTemplate.from_template(
    """
    You are an M&A knowledge assistant and your job is to answer
    the user questions based ONLY on the retrieved context from the M&A Playbook.

    Do not use any outside knowledge and do not make up information.
    If the answer cannot be found in the provided M&A Playbook, say - "The information is not available in the M&A Playbook."

    Retrieved Context:{context}
    Question:{question}

    Provide a concise but complete answer.
    """
)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

In [ ]:
rag_chain=rag_prompt|llm|StrOutputParser()

In [ ]:
def format_context(docs):
  return "\n\n".join([f"[Page {d.metadata['page']+1}]\n{d.page_content}" for d in docs])

def ask(question):
  retrieved_docs=retriever.invoke(question) # retrieve relevant chunks
  context=format_context(retrieved_docs)    # pass context to the LLM
  response=rag_chain.invoke({"context":context,"question":question}) # generate answer
  return response,retrieved_docs

def rag_app(question):
  response,docs=ask(question)
  print("QUESTION:",question,"\n")
  print("ANSWER:",response)
  print("\nRETRIEVED CONTEXT / SOURCES:")
  for d in docs:
    print(f"[Page {d.metadata['page']+1}]",d.page_content.replace("\n"," "),"\n")
  print("="*80)

### Sample queries

In [ ]:
rag_app("What are the core clauses covered in an M&A agreement?")

In [ ]:
rag_app("Which valuation methods are mentioned in the playbook?")

In [ ]:
rag_app("What risks are covered in the risk checklist and how are they mitigated?")

### Question not present in the document

Expected: the system states the information is not available.

In [ ]:
rag_app("What is the capital of France?")

### Interactive Q&A

In [ ]:
# Interactive Q&A (type 'exit' to stop)
while True:
  q=input("Ask the M&A assistant> ").strip()
  if q.lower() in ("exit","quit",""):
    break
  rag_app(q)